# 🍔 AI Food Fraud Detection — Full Pipeline

**Project**: Detecting AI-manipulated/generated food images used for fraudulent refund claims on platforms like UberEats & DoorDash.

**Architecture**: Fine-tuned `google/vit-base-patch16-224-in21k` + Advanced ELA Preprocessing + 8-Feature Forensic Analyzer.

**Dataset**: [CIFAKE: Real and AI-Generated Synthetic Images](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images)

---
```
Pipeline:
  Input Image
       │
       ├── [A] Advanced ELA (multi-quality, JPEG Ghost, spatial blobs)
       ├── [B] 8-Feature Forensic Analyzer (ELA, Noise, Texture, FFT,
       │        Illumination, Shadow, Bokeh, Color) → Fused Forensic Score
       └── [C] ViT Classifier (fine-tuned on CIFAKE + ELA maps)
                │
                └── Weighted Final Verdict + Risk Level
```
---
> ⚡ **Runtime**: Set to `GPU → T4` before running. Runtime → Change runtime type → T4 GPU

## 📦 Cell 1 — Install All Dependencies

In [ ]:
# ============================================================
# CELL 1: Install All Dependencies
# ============================================================
!pip install -q transformers datasets evaluate accelerate
!pip install -q kaggle scikit-learn pillow matplotlib seaborn tqdm
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless scikit-image scipy

import os, io, warnings, random, json, zipfile
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from PIL import Image, ImageChops, ImageEnhance, ImageFilter
from scipy import ndimage
from scipy.stats import entropy
from skimage.feature import local_binary_pattern
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops

from transformers import ViTForImageClassification
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    accuracy_score, ConfusionMatrixDisplay
)

from google.colab import files as colab_files

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
else:
    print('⚠️  No GPU detected. Switch to GPU runtime for much faster training.')

print('\n✅ All dependencies loaded. Ready.')

## 🔑 Cell 2 — Kaggle API Setup & CIFAKE Dataset Download

In [ ]:
# CELL 2 ALTERNATIVE — Hardcode Kaggle credentials (no upload needed)
import os

os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'   # ← paste from kaggle.json
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_API_KEY'    # ← paste from kaggle.json

print("⏳ Downloading CIFAKE...")
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q -o cifake-real-and-ai-generated-synthetic-images.zip -d /content/cifake
!rm -f cifake-real-and-ai-generated-synthetic-images.zip

# Verify
for split in ['train', 'test']:
    for cls in ['REAL', 'FAKE']:
        path = f'/content/cifake/{split}/{cls}'
        count = len(os.listdir(path)) if os.path.exists(path) else 'MISSING'
        print(f"  {path}: {count}")

## 🔬 Cell 3 — Advanced ELA Engine (Fixed Amplifier + JPEG Ghost + Spatial Blob Detection)

In [ ]:
# ============================================================
# CELL 3: Advanced ELA Engine
# ============================================================
# WHY FIXED AMPLIFIER?
#   Old approach: ela_diff → scale = 255/max_diff → enhance(scale)
#   This normalizes the ENTIRE map so the brightest pixel always = 255.
#   A real image with one noisy pixel gets stretched wall-to-wall,
#   completely drowning out any pasted region's true signal.
#
# THE FIX: Fixed amplifier (×12). Preserves relative differences.
#   Manipulated regions stay VISIBLY brighter than authentic background.
#
# JPEG GHOST: Run ELA at multiple quality levels.
#   A pasted element from source quality Q_src will have minimal residual
#   at Q=Q_src but HIGH residual at all other qualities — revealing it.

def ela_at_quality(pil_image: Image.Image, quality: int, amplifier: float = 12.0):
    """Core ELA at a single JPEG quality level with fixed amplification."""
    buf = io.BytesIO()
    pil_image.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    resaved  = Image.open(buf).convert('RGB')
    diff_arr = np.array(ImageChops.difference(pil_image, resaved), dtype=np.float32)
    amplified = np.clip(diff_arr * amplifier, 0, 255).astype(np.uint8)
    ela_pil   = Image.fromarray(amplified)
    raw_diff  = diff_arr.mean(axis=2)
    return amplified.astype(np.float32), ela_pil, raw_diff


def multi_quality_ela(image_path: str, qualities=None, amplifier=12.0):
    """JPEG Ghost technique: run ELA at several quality levels."""
    if qualities is None:
        qualities = [75, 80, 85, 90, 95]
    img = Image.open(image_path).convert('RGB')
    results = {}
    for q in qualities:
        arr, pil, raw = ela_at_quality(img, q, amplifier)
        results[q] = {'array': arr, 'pil': pil, 'raw': raw, 'mean_score': raw.mean()}
    return results


def detect_anomaly_regions(raw_diff: np.ndarray, min_blob_pixels=50):
    """Find spatially localized high-ELA regions using Otsu thresholding + blob analysis."""
    smooth = ndimage.gaussian_filter(raw_diff, sigma=1.5)
    try:
        thresh = threshold_otsu(smooth)
    except Exception:
        thresh = smooth.mean() + smooth.std()

    binary = ndimage.binary_opening(smooth > thresh, iterations=2)
    labeled = label(binary)
    regions = [r for r in regionprops(labeled, intensity_image=smooth)
               if r.area >= min_blob_pixels]

    if not regions:
        return {'threshold': thresh, 'binary_mask': binary, 'labeled': labeled,
                'regions': [], 'top_region': None, 'suspicion_score': 0.0,
                'bg_mean': smooth.mean(), 'top_mean': 0.0, 'z_score': 0.0,
                'compactness': 0.0}

    top = max(regions, key=lambda r: r.mean_intensity)
    bg_pixels = smooth[labeled != top.label]
    bg_mean   = bg_pixels.mean() if len(bg_pixels) else smooth.mean()
    bg_std    = bg_pixels.std()  if len(bg_pixels) else smooth.std()
    z_score   = (top.mean_intensity - bg_mean) / (bg_std + 1e-7)
    compactness = (4 * np.pi * top.area) / ((top.perimeter ** 2) + 1e-7)
    suspicion_score = min(100.0, z_score * 15 + (1 - compactness) * 20)

    return {'threshold': thresh, 'binary_mask': binary, 'labeled': labeled,
            'regions': regions, 'top_region': top, 'suspicion_score': suspicion_score,
            'bg_mean': bg_mean, 'top_mean': top.mean_intensity, 'z_score': z_score,
            'compactness': compactness}


def jpeg_ghost_analysis(image_path: str, qualities=None):
    """Detect quality-inconsistent regions (JPEG Ghost) — reveals pasted objects."""
    if qualities is None:
        qualities = [70, 75, 80, 85, 90, 95]
    img = Image.open(image_path).convert('RGB')
    raw_maps, mean_scores = [], []
    for q in qualities:
        _, _, raw = ela_at_quality(img, q, amplifier=1.0)
        raw_maps.append(raw)
        mean_scores.append(raw.mean())
    raw_stack  = np.stack(raw_maps, axis=0)
    ghost_map  = raw_stack.var(axis=0)
    ghost_quality = qualities[int(np.argmin(mean_scores))]
    smooth_ghost  = ndimage.gaussian_filter(ghost_map, sigma=2.0)
    ghost_thresh  = np.percentile(smooth_ghost.flatten(), 90)
    hot_region    = smooth_ghost > ghost_thresh
    labeled_hot   = label(hot_region)
    hot_regions   = regionprops(labeled_hot)
    max_hot_blob  = max((r.area for r in hot_regions), default=0)
    ghost_flag    = (max_hot_blob / (ghost_map.size) > 0.005) and hot_region.mean() < 0.25
    return ghost_flag, smooth_ghost, ghost_quality, mean_scores, qualities


def get_ela_for_vit(image_path: str, quality=90, amplify=10) -> Image.Image:
    """Returns a 224×224 ELA PIL image suitable for ViT input."""
    orig = Image.open(image_path).convert('RGB')
    buf  = io.BytesIO()
    orig.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    comp = Image.open(buf).convert('RGB')
    diff = ImageChops.difference(orig, comp)
    ela  = np.array(ImageEnhance.Brightness(diff).enhance(amplify))
    return Image.fromarray(ela).resize((224, 224))


def classify_strict(score, z_score, ghost_flag):
    """Classify based on suspicion score, Z-score, and JPEG ghost flag."""
    if score >= 60 or (z_score >= 4.0 and ghost_flag):
        return '🚨 AI-MANIPULATED / FRAUD', 'HIGH', '#E74C3C'
    elif score >= 35 or z_score >= 2.5 or ghost_flag:
        return '⚠️  SUSPICIOUS — Requires Review', 'MEDIUM', '#FF7F0E'
    elif score >= 18 or z_score >= 1.5:
        return '🔶 LOW-MEDIUM SUSPICION', 'LOW-MEDIUM', '#BCBD22'
    else:
        return '✅ AUTHENTIC / GENUINE', 'CLEAR', '#2CA02C'


print('✅ Cell 3: Advanced ELA Engine loaded.')

## 🧬 Cell 4 — 8-Feature Forensic Analyzer

In [ ]:
# ============================================================
# CELL 4: 8-Feature Forensic Analyzer
# ============================================================
# Each feature returns an anomaly score 0–100 (0=Real, 100=AI).
# Fusion weights tuned for food-domain AI detection.

def feat_ela_anomaly(img, quality=90):
    """ELA: Low variance → lack of prior JPEG compression → AI trait."""
    _, encoded = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    ela = cv2.absdiff(img, cv2.imdecode(encoded, 1))
    return max(0, 100 - (np.var(ela) * 2))

def feat_noise_residual(img):
    """Noise: AI generators fail to replicate uniform sensor noise."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    residual = cv2.absdiff(gray, cv2.GaussianBlur(gray, (5, 5), 0))
    return max(0, 100 - (np.std(residual) * 5))

def feat_texture_lbp(img):
    """Texture (LBP): AI food lacks micro-texture → low entropy."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lbp  = local_binary_pattern(gray, 8, 1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), density=True)
    ent = entropy(hist + 1e-7)
    return max(0, 100 - (ent * 30))

def feat_fft_artifacts(img):
    """FFT: Detects frequency anomalies from upsampling networks."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    fshift = np.fft.fftshift(np.fft.fft2(gray))
    mag = 20 * np.log(np.abs(fshift) + 1)
    h, w = gray.shape
    mask = np.ones((h, w), np.uint8)
    cv2.circle(mask, (w // 2, h // 2), int(min(h, w) * 0.25), 0, -1)
    ratio = np.sum(mag * mask) / (np.sum(mag) + 1e-7)
    return max(0, 100 - (ratio * 200))

def feat_illumination_anomaly(img):
    """Illumination: Checks for conflicting light gradients."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    angles = np.arctan2(cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5),
                        cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5))
    hist, _ = np.histogram(angles, bins=36, range=(-np.pi, np.pi))
    peak_ratio = np.max(hist) / (np.sum(hist) + 1e-7)
    return max(0, 100 - (peak_ratio * 500))

def feat_shadow_consistency(img):
    """Shadow: High variance in shadow pixels often indicates AI noise."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, dark_mask = cv2.threshold(gray, 60, 255, cv2.THRESH_BINARY_INV)
    shadow_pixels = gray[dark_mask == 255]
    if len(shadow_pixels) < 100:
        return 0
    return min(100, np.std(shadow_pixels) * 2)

def feat_bokeh_anomaly(img):
    """Bokeh: AI applies bokeh as binary mask, missing natural focal roll-off."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    variance = np.var(cv2.Laplacian(gray, cv2.CV_64F))
    return min(100, abs(variance - 500) / 10)

def feat_color_stats(img):
    """Color: AI food is often over-saturated with uniform chromatic distribution."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]
    hist, _ = np.histogram(saturation.ravel(), bins=256, density=True)
    ent = entropy(hist + 1e-7)
    mean_sat = np.mean(saturation)
    return min(100, (mean_sat / 2.5) + (max(0, 5 - ent) * 10))


FORENSIC_WEIGHTS = {
    'ela'           : 0.10,
    'noise'         : 0.10,
    'texture_lbp'   : 0.15,
    'fft'           : 0.10,
    'illumination'  : 0.10,
    'shadow'        : 0.15,
    'bokeh'         : 0.20,
    'color'         : 0.10,
}

def run_forensic_analysis(image_path: str) -> dict:
    """Run all 8 forensic features and return fused score + per-feature breakdown."""
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        pil = Image.open(image_path).convert('RGB')
        img_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

    scores = {
        'ela'          : feat_ela_anomaly(img_bgr),
        'noise'        : feat_noise_residual(img_bgr),
        'texture_lbp'  : feat_texture_lbp(img_bgr),
        'fft'          : feat_fft_artifacts(img_bgr),
        'illumination' : feat_illumination_anomaly(img_bgr),
        'shadow'       : feat_shadow_consistency(img_bgr),
        'bokeh'        : feat_bokeh_anomaly(img_bgr),
        'color'        : feat_color_stats(img_bgr),
    }
    fused = sum(scores[k] * FORENSIC_WEIGHTS[k] for k in scores)
    return {'scores': scores, 'fused_score': fused}


print('✅ Cell 4: 8-Feature Forensic Analyzer loaded.')

## ⚙️ Cell 5 — Config & Hyperparameters

In [ ]:
# ============================================================
# CELL 5: Config & Hyperparameters
# ============================================================

CONFIG = {
    # Model
    'model_name'        : 'google/vit-base-patch16-224-in21k',
    'num_labels'        : 2,
    'label2id'          : {'AUTHENTIC': 0, 'AI_MANIPULATED': 1},
    'id2label'          : {0: 'AUTHENTIC', 1: 'AI_MANIPULATED'},

    # Dataset paths
    'train_real_dir'    : '/content/cifake/train/REAL',
    'train_fake_dir'    : '/content/cifake/train/FAKE',
    'test_real_dir'     : '/content/cifake/test/REAL',
    'test_fake_dir'     : '/content/cifake/test/FAKE',

    # Data sampling (reduce if Colab RAM is low)
    'max_train_samples' : 10000,   # ← set 5000 if RAM runs out
    'max_test_samples'  : 2000,
    'image_size'        : 224,
    'use_ela'           : True,    # ELA preprocessing before ViT

    # Training
    'batch_size'        : 32,
    'num_epochs'        : 5,       # ← increase to 10 for better accuracy
    'learning_rate'     : 2e-5,
    'weight_decay'      : 0.01,
    'warmup_ratio'      : 0.1,
    'seed'              : 42,
    'output_dir'        : '/content/food_fraud_vit',
    'fp16'              : DEVICE == 'cuda',
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
torch.manual_seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print('=== Food Fraud Detection Config ===')
for k, v in CONFIG.items():
    if k not in ('label2id', 'id2label'):
        print(f'  {k:<22}: {v}')

In [ ]:
# ============================================================
# 🔧 DIAGNOSTIC CELL — Run this BEFORE Cell 6
# ============================================================
import os

print("=== Checking /content/cifake structure ===\n")

if not os.path.exists('/content/cifake'):
    print("❌ /content/cifake does NOT exist — dataset was never downloaded.\n"
          "   → Re-run Cell 2 (Kaggle download) and wait for it to finish fully.")
else:
    for root, dirs, files_ in os.walk('/content/cifake'):
        level = root.replace('/content/cifake', '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 3:
            for f in sorted(files_)[:3]:
                print(f'{indent}  {f}')
    print()
    # Find where REAL images actually are
    for dirpath, dirnames, filenames in os.walk('/content/cifake'):
        if any(f.lower().endswith(('.jpg','.png')) for f in filenames):
            print(f"✅ Images found at: {dirpath}  ({len(filenames)} files)")

## 📂 Cell 6 — Dataset Class & DataLoaders

In [ ]:
# ============================================================
# CELL 6: Dataset Class + DataLoaders
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class FoodFraudDataset(Dataset):
    """Binary food fraud dataset. Label 0=AUTHENTIC, 1=AI_MANIPULATED."""

    def __init__(self, real_dir, fake_dir, transform=None, max_samples=None, use_ela=False):
        self.transform = transform
        self.use_ela   = use_ela
        self.samples   = []

        real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        if max_samples:
            half = max_samples // 2
            real_files = random.sample(real_files, min(half, len(real_files)))
            fake_files = random.sample(fake_files, min(half, len(fake_files)))

        self.samples = [(p, 0) for p in real_files] + [(p, 1) for p in fake_files]
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = get_ela_for_vit(path) if self.use_ela else Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224))
        if self.transform:
            img = self.transform(img)
        return {'pixel_values': img, 'labels': torch.tensor(label, dtype=torch.long)}


train_dataset = FoodFraudDataset(
    CONFIG['train_real_dir'], CONFIG['train_fake_dir'],
    transform=train_transform, max_samples=CONFIG['max_train_samples'],
    use_ela=CONFIG['use_ela']
)
test_dataset = FoodFraudDataset(
    CONFIG['test_real_dir'], CONFIG['test_fake_dir'],
    transform=val_transform, max_samples=CONFIG['max_test_samples'],
    use_ela=CONFIG['use_ela']
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train dataset: {len(train_dataset)} images  |  Test: {len(test_dataset)} images')
print(f'   Train batches : {len(train_loader)}  |  Test batches: {len(test_loader)}')

## 🤖 Cell 7 — ViT Model Setup

In [ ]:
# ============================================================
# CELL 7: Load ViT Model
# ============================================================

model = ViTForImageClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    id2label=CONFIG['id2label'],
    label2id=CONFIG['label2id'],
    ignore_mismatched_sizes=True,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model         : {CONFIG["model_name"]}')
print(f'Total params  : {total_params / 1e6:.1f}M')
print(f'Trainable     : {trainable_params / 1e6:.1f}M')
print(f'Device        : {DEVICE}')
print(f'Classes       : {CONFIG["id2label"]}')

## 🏋️ Cell 8 — Training Loop

In [ ]:
# ============================================================
# CELL 8: Training Loop (AdamW + OneCycleLR + AMP)
# ============================================================
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.notebook import tqdm
import time

optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'],
                        weight_decay=CONFIG['weight_decay'])
scheduler = OneCycleLR(optimizer, max_lr=CONFIG['learning_rate'],
                       steps_per_epoch=len(train_loader),
                       epochs=CONFIG['num_epochs'], pct_start=CONFIG['warmup_ratio'])
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler(enabled=CONFIG['fp16'])

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0


def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in tqdm(loader, desc='Training', leave=False):
        pv = batch['pixel_values'].to(DEVICE)
        lb = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=CONFIG['fp16']):
            out  = model(pixel_values=pv, labels=lb)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total_loss += loss.item() * lb.size(0)
        correct    += (out.logits.argmax(dim=1) == lb).sum().item()
        total      += lb.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            pv = batch['pixel_values'].to(DEVICE)
            lb = batch['labels'].to(DEVICE)
            out = model(pixel_values=pv, labels=lb)
            total_loss += out.loss.item() * lb.size(0)
            correct    += (out.logits.argmax(dim=1) == lb).sum().item()
            total      += lb.size(0)
    return total_loss / total, correct / total


print('=' * 60)
print('🚀 Starting Food Fraud ViT Training')
print(f'   Epochs: {CONFIG["num_epochs"]}  |  Batch: {CONFIG["batch_size"]}  |  LR: {CONFIG["learning_rate"]}')
print(f'   FP16: {CONFIG["fp16"]}  |  ELA Preprocessing: {CONFIG["use_ela"]}')
print('=' * 60)

for epoch in range(CONFIG['num_epochs']):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader)
    va_loss, va_acc = eval_epoch(model, test_loader)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    print(f'Epoch [{epoch+1:02d}/{CONFIG["num_epochs"]}] '
          f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}% | '
          f'Val Loss: {va_loss:.4f}  Acc: {va_acc*100:.2f}% | '
          f'Time: {elapsed:.0f}s')

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        model.save_pretrained(CONFIG['output_dir'])
        print(f'  💾 Best model saved (val_acc={best_val_acc*100:.2f}%)')

print(f'\n✅ Training complete. Best Val Accuracy: {best_val_acc*100:.2f}%')

## 📈 Cell 9 — Training Curves

In [ ]:
# ============================================================
# CELL 9: Training Curves
# ============================================================

epochs_range = range(1, CONFIG['num_epochs'] + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Loss Curves — Food Fraud ViT', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.4)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc')
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc')
axes[1].set_title('Accuracy Curves — Food Fraud ViT', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.4); axes[1].set_ylim([50, 100])

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/training_curves.png')

## 📊 Cell 10 — Full Evaluation: Accuracy, F1, ROC-AUC, EER, APCER, BPCER

In [ ]:
# ============================================================
# CELL 10: Full Evaluation Metrics
# ============================================================

# Load best checkpoint
model_eval = ViTForImageClassification.from_pretrained(
    CONFIG['output_dir'],
    id2label=CONFIG['id2label'],
    label2id=CONFIG['label2id']
).to(DEVICE)
model_eval.eval()

all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Running Evaluation'):
        pv     = batch['pixel_values'].to(DEVICE)
        labels = batch['labels']
        probs  = F.softmax(model_eval(pixel_values=pv).logits, dim=1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(probs.argmax(axis=1))
        all_probs.extend(probs[:, 1])

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

# Core metrics
accuracy    = accuracy_score(all_labels, all_preds)
f1_macro    = f1_score(all_labels, all_preds, average='macro')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')
roc_auc     = roc_auc_score(all_labels, all_probs)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
TN, FP, FN, TP = cm.ravel()

# APCER & BPCER (ISO/IEC 30107-3)
APCER = FN / (TP + FN) if (TP + FN) > 0 else 0.0   # fraud missed as real
BPCER = FP / (TN + FP) if (TN + FP) > 0 else 0.0   # real flagged as fraud
ACER  = (APCER + BPCER) / 2

# EER
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
fnr = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - fnr))
EER = (fpr[eer_idx] + fnr[eer_idx]) / 2

print('=' * 55)
print('  AI FOOD FRAUD DETECTION — EVALUATION REPORT')
print('=' * 55)
print(f'  Accuracy              : {accuracy*100:.2f}%')
print(f'  F1-Score (macro)      : {f1_macro:.4f}')
print(f'  F1-Score (weighted)   : {f1_weighted:.4f}')
print(f'  ROC-AUC               : {roc_auc:.4f}')
print(f'  EER                   : {EER*100:.2f}%')
print('-' * 55)
print(f'  APCER (Fraud→Real miss rate)  : {APCER*100:.2f}%')
print(f'  BPCER (Real→Fraud false alarm): {BPCER*100:.2f}%')
print(f'  ACER  (Average Error Rate)    : {ACER*100:.2f}%')
print('-' * 55)
print(f'  Confusion Matrix: TP={TP}  TN={TN}  FP={FP}  FN={FN}')
print('=' * 55)
print()
print(classification_report(all_labels, all_preds,
                             target_names=['AUTHENTIC', 'AI_MANIPULATED']))

# Visualization — ROC + Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='#2980B9', lw=2, label=f'ROC (AUC={roc_auc:.4f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1, label='Random')
axes[0].scatter(fpr[eer_idx], tpr[eer_idx], color='red', zorder=5, s=80,
                label=f'EER={EER*100:.2f}%')
axes[0].set_title('ROC Curve — AI Food Fraud Detection', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (BPCER)')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(); axes[0].grid(alpha=0.4)

ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['AUTHENTIC','AI_MANIPULATED']).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix', fontweight='bold')

plt.suptitle(f'Accuracy: {accuracy*100:.2f}%  |  AUC: {roc_auc:.4f}  |  EER: {EER*100:.2f}%',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/evaluation_results.png')

## 💾 Cell 11 — Save Model & Metrics Report

In [ ]:
# ============================================================
# CELL 11: Save Model + Export Metrics JSON
# ============================================================

SAVE_DIR = '/content/food_fraud_final'
os.makedirs(SAVE_DIR, exist_ok=True)
model_eval.save_pretrained(SAVE_DIR)

metrics_report = {
    'project'   : 'AI Food Fraud Detection',
    'model'     : CONFIG['model_name'],
    'dataset'   : 'CIFAKE (birdy654)',
    'timestamp' : datetime.now().isoformat(),
    'config': {
        'epochs'       : CONFIG['num_epochs'],
        'batch_size'   : CONFIG['batch_size'],
        'lr'           : CONFIG['learning_rate'],
        'ela_enabled'  : CONFIG['use_ela'],
        'train_samples': CONFIG['max_train_samples'],
        'test_samples' : CONFIG['max_test_samples'],
    },
    'metrics': {
        'accuracy'        : round(accuracy, 4),
        'f1_macro'        : round(f1_macro, 4),
        'f1_weighted'     : round(f1_weighted, 4),
        'roc_auc'         : round(roc_auc, 4),
        'eer'             : round(float(EER), 4),
        'apcer'           : round(float(APCER), 4),
        'bpcer'           : round(float(BPCER), 4),
        'acer'            : round(float(ACER), 4),
        'confusion_matrix': {'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN)},
    },
    'training_history': history,
}

report_path = os.path.join(SAVE_DIR, 'metrics_report.json')
with open(report_path, 'w') as f:
    json.dump(metrics_report, f, indent=2)

print('✅ Model saved to:', SAVE_DIR)
print('✅ Metrics report saved to:', report_path)
print()
print('=== FINAL SUMMARY ===')
print(f'  Accuracy : {accuracy*100:.2f}%')
print(f'  ROC-AUC  : {roc_auc:.4f}')
print(f'  EER      : {EER*100:.2f}%')
print(f'  APCER    : {APCER*100:.2f}%')
print(f'  BPCER    : {BPCER*100:.2f}%')

---
## 🖼️ Cell 12 — Upload & Test Your Own Food Images

Upload **any food image(s)** from your device. Each image gets:
- **ViT prediction** (AUTHENTIC / AI_MANIPULATED + confidence)
- **Advanced ELA** (multi-quality, JPEG Ghost, spatial blob detection)
- **8-Feature Forensic Score** (fused from ELA, Noise, Texture, FFT, Illumination, Shadow, Bokeh, Color)
- **Final weighted verdict** combining all three analyses

> ⚠️ Run Cells 1–10 first. If reloading a saved model, run Cells 1, 3, 4, 5, then jump here.

In [ ]:
# ============================================================
# CELL 12 — STEP 1: Upload Images
# ============================================================

UPLOAD_DIR = '/content/custom_uploads'
RESULT_DIR = '/content/custom_results'
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print('📂 Select one or more food images (JPG / PNG)...')
uploaded = colab_files.upload()

if not uploaded:
    raise RuntimeError('No files uploaded — re-run and select at least one image.')

upload_paths = []
for fname, data in uploaded.items():
    p = os.path.join(UPLOAD_DIR, fname)
    with open(p, 'wb') as f:
        f.write(data)
    upload_paths.append(p)

print(f'\n✅ {len(upload_paths)} file(s) ready: {[os.path.basename(p) for p in upload_paths]}')
print('▶️  Run the next cell to classify them.')

In [ ]:
# ============================================================
# CELL 12 — STEP 2: Classify Each Image (Full 3-Layer Analysis)
# ============================================================

THRESHOLD = 0.5   # ViT threshold — lower to 0.3 to flag more borderline cases
USE_ELA   = True  # Use ELA preprocessing for ViT input

_infer_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

ELA_QUALITIES = [75, 80, 85, 90, 95]
ELA_AMPLIFIER = 12.0
MIN_BLOB_PX   = 40


def _risk_label(p):
    if p < 0.30: return '🟢 LOW RISK'
    if p < 0.60: return '🟡 MEDIUM RISK'
    return '🔴 HIGH RISK'


def full_inference(image_path: str, model, threshold=0.5, use_ela=True):
    """
    3-layer analysis:
      Layer 1: ViT classifier (primary ML verdict)
      Layer 2: Advanced ELA + JPEG Ghost + spatial blob suspicion
      Layer 3: 8-feature forensic score
    """
    fname = os.path.basename(image_path)
    print(f'\n{"="*65}')
    print(f'  🔍 ANALYSING: {fname}')
    print(f'{"="*65}')

    # ── Layer 1: ViT Prediction ────────────────────────────────────
    print('  [1/3] ViT classifier...')
    try:
        src = get_ela_for_vit(image_path) if use_ela else Image.open(image_path).convert('RGB').resize((224,224))
    except Exception:
        src = Image.open(image_path).convert('RGB').resize((224,224))
    tensor = _infer_tf(src).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(pixel_values=tensor).logits, dim=1).cpu().numpy()[0]
    fraud_p, real_p = float(probs[1]), float(probs[0])
    vit_pred = 'AI_MANIPULATED' if fraud_p >= threshold else 'AUTHENTIC'
    vit_conf = max(fraud_p, real_p)
    print(f'      ViT verdict: {vit_pred} ({vit_conf:.1%} confidence)')

    # ── Layer 2: Advanced ELA + JPEG Ghost ────────────────────────
    print('  [2/3] Advanced ELA + JPEG Ghost...')
    ela_results = multi_quality_ela(image_path, ELA_QUALITIES, ELA_AMPLIFIER)
    primary_raw = ela_results[85]['raw']
    anomaly     = detect_anomaly_regions(primary_raw, min_blob_pixels=MIN_BLOB_PX)
    ghost_flag, ghost_map, ghost_q, _, _ = jpeg_ghost_analysis(image_path, ELA_QUALITIES)
    ela_verdict, ela_risk, ela_color = classify_strict(
        anomaly['suspicion_score'], anomaly['z_score'], ghost_flag
    )
    print(f'      ELA suspicion score: {anomaly["suspicion_score"]:.1f}/100 | Z={anomaly["z_score"]:.2f} | Ghost={"YES" if ghost_flag else "NO"}')

    # ── Layer 3: 8-Feature Forensic ────────────────────────────────
    print('  [3/3] 8-feature forensic analysis...')
    forensic = run_forensic_analysis(image_path)
    forensic_score = forensic['fused_score']
    print(f'      Forensic fused score: {forensic_score:.1f}/100')

    # ── Final Weighted Verdict ─────────────────────────────────────
    # ViT has highest weight (most powerful), ELA medium, forensic as support
    fraud_score = (fraud_p * 100 * 0.55) + \
                  (anomaly['suspicion_score'] * 0.25) + \
                  (forensic_score * 0.20)

    if fraud_score >= 60:
        final_verdict = '🚨 AI-GENERATED / FRAUD'
        final_color   = '#E74C3C'
    elif fraud_score >= 35:
        final_verdict = '⚠️  SUSPICIOUS'
        final_color   = '#FF7F0E'
    else:
        final_verdict = '✅ AUTHENTIC'
        final_color   = '#2CA02C'

    risk = _risk_label(fraud_p)

    # ── 6-Panel Figure ─────────────────────────────────────────────
    original = Image.open(image_path).convert('RGB')
    ela_arr  = np.array(ela_results[85]['pil'])

    fig = plt.figure(figsize=(20, 11))
    fig.patch.set_facecolor('#0F0F0F')
    gs  = plt.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.12)

    def dark_ax(ax, title, color='white'):
        ax.set_title(title, color=color, fontsize=10, fontweight='bold', pad=7)
        ax.axis('off')
        ax.set_facecolor('#1A1A1A')

    # Panel 1 — Original + verdict
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(original)
    if anomaly['top_region'] is not None:
        minr, minc, maxr, maxc = anomaly['top_region'].bbox
        ax1.add_patch(mpatches.Rectangle((minc, minr), maxc-minc, maxr-minr,
                      linewidth=3, edgecolor='red', facecolor='none', linestyle='--'))
        ax1.text(minc, max(minr-5,0), '⬆ Suspect Region', color='red', fontsize=8, fontweight='bold')
    dark_ax(ax1, f'{fname}\n{final_verdict}\nFraud Score: {fraud_score:.1f}/100  |  {risk}', color=final_color)

    # Panel 2 — ELA Q=85
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(ela_arr)
    dark_ax(ax2, f'ELA Map (Q=85, ×{ELA_AMPLIFIER:.0f} fixed)\nSuspicion: {anomaly["suspicion_score"]:.1f}/100')

    # Panel 3 — ELA Q=90
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(np.array(ela_results[90]['pil']))
    dark_ax(ax3, f'ELA Map (Q=90)\nJPEG Ghost: {"YES 🚨" if ghost_flag else "NO ✅"}  |  Source Q≈{ghost_q}')

    # Panel 4 — ELA heatmap with blob
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.imshow(primary_raw, cmap='inferno', interpolation='bilinear')
    if anomaly['top_region'] is not None:
        minr, minc, maxr, maxc = anomaly['top_region'].bbox
        ax4.add_patch(mpatches.Rectangle((minc, minr), maxc-minc, maxr-minr,
                      linewidth=2, edgecolor='cyan', facecolor='none'))
    dark_ax(ax4, f'Spatial Blob Detection\nZ-Score: {anomaly["z_score"]:.2f}  |  BG ELA: {anomaly["bg_mean"]:.3f}')

    # Panel 5 — Forensic feature bar chart
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.set_facecolor('#1A1A1A')
    feat_names = list(forensic['scores'].keys())
    feat_vals  = [forensic['scores'][k] for k in feat_names]
    bar_colors = ['#E74C3C' if v > 60 else '#FF7F0E' if v > 35 else '#2ECC71' for v in feat_vals]
    bars = ax5.barh(feat_names, feat_vals, color=bar_colors, edgecolor='white', linewidth=0.4)
    ax5.set_xlim(0, 100)
    ax5.axvline(60, color='red', linestyle='--', lw=1, alpha=0.6)
    ax5.axvline(35, color='orange', linestyle='--', lw=1, alpha=0.6)
    ax5.tick_params(colors='white', labelsize=8)
    ax5.spines[:].set_color('#444')
    ax5.set_title(f'8-Feature Forensic Scores\nFused: {forensic_score:.1f}/100',
                  color='white', fontsize=10, fontweight='bold', pad=7)
    for bar, val in zip(bars, feat_vals):
        ax5.text(min(val+1, 93), bar.get_y()+bar.get_height()/2,
                 f'{val:.0f}', va='center', color='white', fontsize=8)

    # Panel 6 — ViT probability bars
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.set_facecolor('#1A1A1A')
    bars6 = ax6.barh(['AUTHENTIC', 'AI_MANIPULATED'], [real_p, fraud_p],
                     color=['#2ECC71','#E74C3C'], edgecolor='white', linewidth=0.5, height=0.5)
    ax6.set_xlim(0, 1)
    ax6.axvline(threshold, color='yellow', linestyle='--', lw=1.5, label=f'Threshold={threshold}')
    ax6.legend(fontsize=8, facecolor='#222', labelcolor='white')
    ax6.tick_params(colors='white')
    ax6.spines[:].set_color('#444')
    ax6.set_title(f'ViT Fraud Probability\n{vit_pred} ({vit_conf:.1%})',
                  color='white', fontsize=10, fontweight='bold', pad=7)
    for bar, val in zip(bars6, [real_p, fraud_p]):
        ax6.text(min(val+0.02, 0.82), bar.get_y()+bar.get_height()/2,
                 f'{val:.1%}', va='center', color='white', fontweight='bold', fontsize=11)

    plt.suptitle(f'FORENSIC REPORT — {fname}',
                 fontsize=13, fontweight='bold', color='white', y=1.01)

    out_path = os.path.join(RESULT_DIR, f'report_{os.path.splitext(fname)[0]}.png')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

    # Console summary
    print(f'\n  ┌─ SUMMARY ──────────────────────────────────────────┐')
    print(f'  │  ViT Prediction    : {vit_pred:<32}│')
    print(f'  │  ViT Fraud Prob    : {fraud_p:.4f} ({fraud_p:.1%}){" "*17}│')
    print(f'  │  ELA Suspicion     : {anomaly["suspicion_score"]:6.1f}/100{" "*22}│')
    print(f'  │  ELA Z-Score       : {anomaly["z_score"]:6.3f}{" "*27}│')
    print(f'  │  JPEG Ghost Flag   : {"YES 🚨" if ghost_flag else "NO  ✅"}{" "*28}│')
    print(f'  │  Forensic Score    : {forensic_score:6.1f}/100{" "*22}│')
    print(f'  │  Final Score       : {fraud_score:6.1f}/100{" "*22}│')
    print(f'  │  FINAL VERDICT     : {final_verdict:<32}│')
    print(f'  │  RISK LEVEL        : {risk:<32}│')
    print(f'  └────────────────────────────────────────────────────┘')

    return {
        'filename'        : fname,
        'vit_prediction'  : vit_pred,
        'fraud_prob'      : round(fraud_p, 4),
        'real_prob'       : round(real_p, 4),
        'confidence'      : round(vit_conf, 4),
        'ela_suspicion'   : round(anomaly['suspicion_score'], 2),
        'ela_z_score'     : round(anomaly['z_score'], 3),
        'jpeg_ghost'      : ghost_flag,
        'forensic_score'  : round(forensic_score, 2),
        'forensic_detail' : {k: round(v, 2) for k, v in forensic['scores'].items()},
        'final_score'     : round(fraud_score, 2),
        'final_verdict'   : final_verdict,
        'risk_level'      : risk,
    }


# ── Run on all uploaded images ─────────────────────────────────────────────
all_results = []
for img_path in upload_paths:
    try:
        result = full_inference(img_path, model_eval, threshold=THRESHOLD, use_ela=USE_ELA)
        all_results.append(result)
    except Exception as e:
        print(f'  ⚠️  Skipped {os.path.basename(img_path)}: {e}')

print(f'\n\n✅ Analysed {len(all_results)} image(s). Run the next cell for the batch dashboard.')

In [ ]:
# ============================================================
# CELL 12 — STEP 3: Batch Dashboard (auto-shown for 2+ images)
# ============================================================

if len(all_results) > 1:
    n_auth  = sum(1 for r in all_results if 'AUTHENTIC' in r['final_verdict'])
    n_fraud = len(all_results) - n_auth
    labels  = [r['filename'][:18] + '…' if len(r['filename']) > 18
               else r['filename'] for r in all_results]
    fraud_scores = [r['final_score'] for r in all_results]

    fig = plt.figure(figsize=(18, 10))
    fig.patch.set_facecolor('#0F0F0F')
    gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)
    ax_pie = fig.add_subplot(gs[0, 0])
    ax_bar = fig.add_subplot(gs[0, 1:])
    ax_tbl = fig.add_subplot(gs[1, :])
    for ax in [ax_pie, ax_bar, ax_tbl]:
        ax.set_facecolor('#1A1A1A')

    # Pie chart
    sizes   = [s for s in [n_auth, n_fraud] if s > 0]
    plabels = [l for l, s in zip(['AUTHENTIC', 'AI_MANIPULATED'], [n_auth, n_fraud]) if s > 0]
    ax_pie.pie(sizes, labels=plabels, colors=['#2ECC71','#E74C3C'],
               autopct='%1.0f%%', startangle=90,
               textprops={'color':'white','fontsize':9})
    ax_pie.set_title('Classification Split', color='white', fontweight='bold')

    # Bar chart — final fraud scores
    bar_cols = ['#E74C3C' if s >= 60 else '#FF7F0E' if s >= 35 else '#2ECC71'
                for s in fraud_scores]
    bars = ax_bar.bar(labels, fraud_scores, color=bar_cols, edgecolor='white', linewidth=0.4)
    ax_bar.axhline(60, color='red',    linestyle='--', lw=1.5, label='Fraud threshold (60)')
    ax_bar.axhline(35, color='orange', linestyle='--', lw=1.2, label='Suspicious threshold (35)')
    ax_bar.set_ylim(0, 100)
    ax_bar.set_ylabel('Final Fraud Score /100', color='white')
    ax_bar.set_title('Per-Image Final Fraud Score', color='white', fontweight='bold')
    ax_bar.tick_params(colors='white')
    ax_bar.spines[:].set_color('#444')
    ax_bar.legend(fontsize=8, facecolor='#222', labelcolor='white')
    plt.setp(ax_bar.get_xticklabels(), rotation=25, ha='right', fontsize=8, color='white')
    for bar, val in zip(bars, fraud_scores):
        ax_bar.text(bar.get_x()+bar.get_width()/2, min(val+1, 92), f'{val:.1f}',
                    ha='center', color='white', fontsize=8)

    # Results table
    ax_tbl.axis('off')
    col_labels = ['File', 'ViT Verdict', 'Fraud %', 'ELA Score', 'Forensic', 'Final Score', 'Risk']
    row_data = [
        [r['filename'], r['vit_prediction'],
         f"{r['fraud_prob']:.1%}",
         f"{r['ela_suspicion']:.1f}",
         f"{r['forensic_score']:.1f}",
         f"{r['final_score']:.1f}",
         r['risk_level']]
        for r in all_results
    ]
    tbl = ax_tbl.table(cellText=row_data, colLabels=col_labels, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.7)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_facecolor('#2C3E50' if row == 0 else '#1A1A1A')
        cell.set_text_props(color='white')
        cell.set_edgecolor('#444')

    plt.suptitle(f'BATCH RESULTS — {len(all_results)} images  |  '
                 f'Authentic: {n_auth}  |  AI/Fraud: {n_fraud}',
                 fontsize=13, fontweight='bold', color='white')
    dashboard_path = os.path.join(RESULT_DIR, 'batch_dashboard.png')
    plt.savefig(dashboard_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'✅ Dashboard saved: {dashboard_path}')
else:
    print('ℹ️  Only 1 image was analysed — batch dashboard requires 2+ images.')

In [ ]:
# ============================================================
# CELL 12 — STEP 4: Save JSON Report + Download ZIP
# ============================================================
import numpy as np

# Custom encoder to handle numpy types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        return super().default(obj)

report = {
    'run_timestamp'  : datetime.now().isoformat(),
    'model'          : CONFIG['model_name'],
    'threshold'      : THRESHOLD,
    'ela_used'       : USE_ELA,
    'total_images'   : len(all_results),
    'authentic'      : sum(1 for r in all_results if 'AUTHENTIC' in r['final_verdict']),
    'ai_manipulated' : sum(1 for r in all_results if 'FRAUD' in r['final_verdict'] or 'SUSPICIOUS' in r['final_verdict']),
    'results'        : all_results,
}

json_path = os.path.join(RESULT_DIR, 'custom_image_results.json')
with open(json_path, 'w') as f:
    json.dump(report, f, indent=2, cls=NumpyEncoder)   # ← NumpyEncoder added

zip_path = '/content/food_fraud_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fn in os.listdir(RESULT_DIR):
        zf.write(os.path.join(RESULT_DIR, fn), fn)
    for extra in ['/content/training_curves.png', '/content/evaluation_results.png']:
        if os.path.exists(extra):
            zf.write(extra, os.path.basename(extra))

print('\n📦 Summary')
print(f'  Images analysed  : {len(all_results)}')
print(f'  AUTHENTIC        : {report["authentic"]}')
print(f'  FRAUD/SUSPICIOUS : {report["ai_manipulated"]}')

colab_files.download(zip_path)
print('\n✅ ZIP downloaded!')

---
## 📋 Quick Reference

| Cell | What it does |
|------|--------------|
| 1 | Install all packages (PyTorch, Transformers, OpenCV, scikit-image, etc.) |
| 2 | Kaggle API setup + CIFAKE dataset download (~1.2 GB) |
| 3 | Advanced ELA engine: fixed amplifier + JPEG Ghost + spatial blob detection |
| 4 | 8-Feature forensic analyzer (ELA, Noise, Texture-LBP, FFT, Illumination, Shadow, Bokeh, Color) |
| 5 | Config (tune `max_train_samples`, `num_epochs`, `use_ela`, `batch_size`) |
| 6 | `FoodFraudDataset` class + DataLoaders |
| 7 | ViT model load from HuggingFace (`google/vit-base-patch16-224-in21k`) |
| 8 | Training loop (AdamW + OneCycleLR + mixed precision AMP) |
| 9 | Loss & accuracy plots |
| 10 | Accuracy, F1, ROC-AUC, EER, APCER, BPCER + ROC curve + confusion matrix |
| 11 | Save best model + export metrics JSON |
| **12 Step 1** | **Upload your own food images** |
| **12 Step 2** | **3-layer analysis: ViT + Advanced ELA + 8-Feature Forensic → 6-panel report** |
| **12 Step 3** | **Batch dashboard: pie chart, bar chart, results table** |
| **12 Step 4** | **Save JSON report + download ZIP of all results** |

### ⚙️ Tuning Tips
- **Low RAM?** → `max_train_samples: 5000`, `max_test_samples: 1000`
- **Better accuracy?** → `num_epochs: 10`, or `use_ela: False` for raw images
- **Catch more fraud?** → Lower `THRESHOLD` to `0.3` in Step 2
- **Skip training?** → After Cell 1+3+4+5, jump to Cell 12 and load from `food_fraud_final/`